# 🦅 FALCON CFE — Entrenar con dataset FINO de Roboflow

Este notebook entrena el modelo con tu **dataset etiquetado a mano** (cajas por sección), descargándolo directo de Roboflow.

**Antes de correr:** activa la GPU (Menú *Entorno de ejecución → Cambiar tipo* → **GPU T4**).

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Conectar Google Drive (para guardar el modelo)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/falcon_cfe'
os.makedirs(BASE, exist_ok=True)
print('Guardando en:', BASE)

## 3. Instalar dependencias

In [ ]:
!pip install -q ultralytics roboflow

## 4. Descargar dataset de Roboflow

**Reemplaza** con el código que Roboflow te da al exportar (botón *Export → YOLO → Show download code*).
El formato es parecido a esto:

In [ ]:
# ============================================================
# PEGA AQUÍ TU CÓDIGO DE DESCARGA DE ROBOFLOW
# (Roboflow te lo da al exportar: Export → Format: YOLOv8 → Show download code)
# Se ve algo así:
# ============================================================

from roboflow import Roboflow
rf = Roboflow(api_key="TU_API_KEY_AQUÍ")  # ← Cambia esto
project = rf.workspace("TU_WORKSPACE").project("falcon-cfe-satelital")  # ← Cambia esto
version = project.version(1)  # ← Cambia al número de versión que exportaste
dataset = version.download("yolov8")  # Descarga en formato YOLO

print('Dataset descargado en:', dataset.location)

## 5. Entrenar (imgsz=960, cajas finas)

Con 500-800 imágenes bien etiquetadas y GPU T4, ~150 épocas tardan 2-3h.

In [ ]:
from ultralytics import YOLO
import yaml

# Leer la ruta del dataset.yaml generado por Roboflow
data_yaml = dataset.location + '/data.yaml'
print('Usando:', data_yaml)

model = YOLO('yolov8m.pt')
results = model.train(
    data=data_yaml,
    epochs=150,
    imgsz=960,
    batch=-1,           # auto (ajusta a la memoria de la GPU)
    device=0,
    name='cfe_fino',
    project=BASE + '/runs',
    save_period=10,     # checkpoint cada 10 épocas (por si se desconecta)
    patience=30,
    # Augmentaciones para satelital:
    degrees=180,        # rotaciones completas (satélite no tiene orientación fija)
    fliplr=0.5,
    flipud=0.5,
    mosaic=1.0,
    scale=0.5,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
)
print('✅ Entrenamiento terminado.')

## 6. Exportar a ONNX y descargar

In [ ]:
import glob, shutil
from google.colab import files

# Exportar el mejor modelo a ONNX
best = glob.glob(BASE + '/runs/**/best.pt', recursive=True)
if best:
    model = YOLO(best[-1])
    onnx_path = model.export(format='onnx', imgsz=960, opset=12, simplify=True)
    print('Modelo ONNX:', onnx_path)
    shutil.copy(onnx_path, 'cfe_satelital_fino.onnx')
    files.download('cfe_satelital_fino.onnx')
else:
    print('No se encontró best.pt')

## 7. Ver resultados

In [ ]:
import glob
from IPython.display import Image, display
for img in glob.glob(BASE + '/runs/**/results.png', recursive=True):
    display(Image(filename=img))
for img in glob.glob(BASE + '/runs/**/val_batch0_pred.jpg', recursive=True)[:3]:
    display(Image(filename=img))

---
## 🔄 REANUDAR (si Colab se desconectó)

Corre: celdas 1→2→3, luego la de abajo:

In [ ]:
from ultralytics import YOLO
import glob
last = glob.glob('/content/drive/MyDrive/falcon_cfe/runs/**/last.pt', recursive=True)
if last:
    model = YOLO(last[-1])
    model.train(resume=True)
else:
    print('No hay checkpoint para reanudar')